In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# exp01_01 ルートをパスに追加
sys.path.append(os.path.abspath('..'))
from configs.config import *
from src.runner import Runner
from src.model_LGBM import model_LGBM
from src.util import Logger, Util

In [ ]:
os.makedirs(DIR_LOG, exist_ok=True)
logger = Logger(path=DIR_LOG)

def get_run_name(model_type):
    suffix = '_' + datetime.now().strftime('%Y%m%d%H%M')
    return model_type + suffix

# 学習データの準備

In [ ]:
# 主キー・目的変数・特徴量を結合
df_all = (
    Util.load_feature('Key')
    .merge(Util.load_feature('Target'), how='left', on=['社員番号', 'category'])
    .merge(Util.load_feature('CategoryFeature'), how='left', on=['社員番号', 'category'])
    .merge(Util.load_feature('UdemyActivityFeature'), how='left', on='社員番号')
    .merge(Util.load_feature('OvertimeWorkByMonthFeature'), how='left', on='社員番号')
    .merge(Util.load_feature('PositionHistoryFeature'), how='left', on='社員番号')
    .merge(Util.load_feature('DxFeature'), how='left', on='社員番号')
    .merge(Util.load_feature('HrFeature'), how='left', on='社員番号')
    .merge(Util.load_feature('CareerFeature'), how='left', on='社員番号')
)

In [ ]:
# train / test に分割
df_train = df_all[df_all['target'].notnull()].reset_index(drop=True)
df_test  = df_all[df_all['target'].isnull()].reset_index(drop=True)
print(f'train shape: {df_train.shape}')
print(f'test  shape: {df_test.shape}')

# LightGBM

In [ ]:
memo = 'exp01_01: ベースライン（多テーブル集計特徴量 + LightGBM + StratifiedGroupKFold 5-fold）'

run_name = get_run_name(model_type='lgbm')
run_name

In [ ]:
def after_predict_process(df_pred, target_col):
    return df_pred

def after_split_process(tr, va):
    return tr, va

model_params_lgb = {
    #### run params
    'key_cols': KEY_COL,
    'target_col': TARGET_COL,
    'remove_cols': [],
    'tune': [False, 0],           # チューニングしない（ベースライン）
    #### model train params
    'num_boost_round': 10000,
    'early_stopping_rounds': 100,
    'verbose': -1,
    'period': 100,
    'verbosity': -1,
    #### model core params
    'boosting_type': 'gbdt',
    'objective': 'binary',
    'metric': 'binary_logloss',
    'is_unbalance': True,          # 不均衡データ対応
    'max_depth': -1,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_freq': 1,
    'learning_rate': 0.05,
    'bagging_fraction': 0.8,
    'random_state': 42,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'min_data_in_leaf': 20,
    'device': 'cpu',
}

run_setting = {
    'after_predict_process': after_predict_process,
    'after_split_process': after_split_process,
}

cv_setting = {
    'target_col': TARGET_COL,
    'group_col': '社員番号',
    'n_splits': 5,
    'shuffle': True,
    'random_state': 42,
}

In [ ]:
ml_runner = Runner(
    run_name,
    model_LGBM,
    model_params_lgb,
    df_train,
    df_test,
    run_setting,
    cv_setting,
    logger,
    memo,
)

In [ ]:
ml_runner.run_train_cv()

In [ ]:
ml_runner.run_metric_cv()

In [ ]:
ml_runner.run_predict_cv()

In [ ]:
ml_runner.plot_feature_importance_cv()

# Submission の作成

In [ ]:
os.makedirs(DIR_SUBMISSIONS, exist_ok=True)

df_te_pred = pd.read_pickle(os.path.join(ml_runner.out_dir_name, 'te_pred.pkl'))
df_prep_test = pd.read_pickle(os.path.join(DIR_INTERIM, 'df_prep_test.pkl'))
df_pred = pd.merge(df_prep_test, df_te_pred, on=['社員番号', 'category'], how='left')
df_submit = df_pred[['target']]

path_submit = os.path.join(DIR_SUBMISSIONS, f'{ml_runner.run_name}_submission.csv')
df_submit.to_csv(path_submit, header=True, index=False)
print(f'submission saved: {path_submit}')

# モデル分析

In [ ]:
# 予測値の統計
df_submit['target'].describe()

In [ ]:
# 予測値の分布
df_submit['target'].hist(bins=50)
plt.title('Prediction distribution')
plt.xlabel('predicted probability')
plt.show()

In [ ]:
# category ごとの AUC
df_va_pred = pd.read_pickle(os.path.join(ml_runner.out_dir_name, 'va_pred.pkl'))
df_va_target = pd.merge(
    df_va_pred,
    Util.load_feature('Target'),
    on=['社員番号', 'category'],
    how='left',
    suffixes=('_pred', '_true'),
)

auc_scores = {}
auc_scores['all'] = ml_runner.metric(
    df_va_target['target_true'], df_va_target['target_pred']
)
for category in df_va_target['category'].unique():
    df_cat = df_va_target[df_va_target['category'] == category]
    auc_scores[category] = ml_runner.metric(df_cat['target_true'], df_cat['target_pred'])

pd.DataFrame(data=auc_scores, index=[0]).T.rename(columns={0: 'AUC'})